# 06. エージェントパターン：Tool Use・ReAct・マルチエージェント

LLMは単なる「質問応答機」から、**自律的に行動するエージェント** へと進化しています。
このノートブックでは、LLMエージェントの主要なパターンを実装しながら学びます。

**このノートブックで学ぶこと:**
- Tool Use（ツール呼び出し）：LLMが外部関数を呼び出す
- ReActパターン：思考→行動→観察のループ
- メモリと状態管理
- マルチエージェント：複数のエージェントが協力する

### LLMエージェントとは？

```
従来のLLM（単純な質問応答）
┌──────────┐  質問  ┌─────────┐  回答  ┌──────────┐
│  ユーザー │ ─────→ │  LLM    │ ─────→ │  ユーザー │
└──────────┘        └─────────┘        └──────────┘
    1回のやり取りで完結

LLMエージェント
┌──────────┐ タスク ┌─────────────────────────────────┐
│  ユーザー │ ─────→ │            エージェント            │
└──────────┘        │  ┌─────────┐  ┌─────────────┐  │
                    │  │   LLM   │  │    Tools    │  │
                    │  │（思考）  │←→│ (web, code, │  │
                    │  └─────────┘  │  DB, API）  │  │
                    │        ↑↓     └─────────────┘  │
                    │  ┌─────────┐  ┌─────────────┐  │
                    │  │ Memory  │  │  Planning   │  │
                    │  └─────────┘  └─────────────┘  │
                    └────────────────────┬────────────┘
                                         │ 最終回答
                    ┌──────────┐  ←─────┘
                    │  ユーザー │
                    └──────────┘
    複数ステップ、ツール活用、自律的な判断
```

**エージェントの構成要素:**
1. **LLM（脳）**: 推論・計画・判断を行う
2. **ツール（手足）**: 検索、計算、コード実行、API呼び出しなど
3. **メモリ（記憶）**: 短期記憶（会話履歴）と長期記憶（ベクターDB）
4. **プランニング（思考）**: タスクをステップに分解する能力

In [ ]:
# インポートとセットアップ
import json
import re
import math
import time
from typing import Any, Optional
from dataclasses import dataclass, field

# Ollama クライアント
try:
    from ollama import Client
    client = Client(host="http://localhost:11434")
    models = client.list()
    available = [m.model for m in models.models]
    print(f"Ollama接続OK。利用可能モデル: {available}")
    OLLAMA_AVAILABLE = True
except Exception as e:
    print(f"Ollama接続失敗: {e}")
    print("デモモードで実行します（実際のLLM呼び出しはスキップ）")
    OLLAMA_AVAILABLE = False
    available = []

# 使用モデル
MODEL = "qwen2.5:7b" if "qwen2.5:7b" in available else (
    available[0] if available else "qwen2.5:7b"
)
print(f"使用モデル: {MODEL}")


def llm_call(messages: list, model: str = None, temperature: float = 0.1) -> str:
    """LLMを呼び出してテキストを返す汎用関数"""
    if model is None:
        model = MODEL

    if not OLLAMA_AVAILABLE:
        return f"[デモ] LLM呼び出しをシミュレートしています。メッセージ数: {len(messages)}"

    response = client.chat(
        model=model,
        messages=messages,
        options={"temperature": temperature}
    )
    return response.message.content


print("セットアップ完了")

## 1. Tool Use（関数呼び出し）

LLMにツール（Python関数）を「説明」し、LLMが必要なときにツールを選んで呼び出せるようにします。

```
ユーザー: 「東京の天気は？」
    ↓
LLM: 「天気を調べるには get_weather ツールが必要」
     → {"tool": "get_weather", "args": {"city": "東京"}}
    ↓
Python: get_weather("東京") → "晴れ、25℃"
    ↓
LLM: 「東京の天気は晴れで気温25℃です。」
```

In [ ]:
# ツール定義（実際の実装）

def get_weather_mock(city: str) -> str:
    """天気情報を取得する（モック実装）"""
    weather_data = {
        "東京": {"condition": "晴れ", "temp": 22, "humidity": 55},
        "大阪": {"condition": "曇り", "temp": 19, "humidity": 68},
        "札幌": {"condition": "雪", "temp": -3, "humidity": 80},
        "福岡": {"condition": "雨", "temp": 16, "humidity": 85},
        "沖縄": {"condition": "晴れ", "temp": 28, "humidity": 75},
    }
    # デフォルト: 未知の都市
    default = {"condition": "不明", "temp": 20, "humidity": 60}
    data = weather_data.get(city, default)
    return f"{city}の天気: {data['condition']}, 気温{data['temp']}℃, 湿度{data['humidity']}%"


def calculate(expression: str) -> str:
    """数式を計算する（安全な評価）"""
    try:
        # 安全な数学関数のみ許可
        allowed_names = {
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow,
            "sqrt": math.sqrt, "floor": math.floor, "ceil": math.ceil,
            "sin": math.sin, "cos": math.cos, "tan": math.tan,
            "pi": math.pi, "e": math.e,
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"計算結果: {expression} = {result}"
    except Exception as ex:
        return f"計算エラー: {str(ex)}"


def search_mock(query: str) -> str:
    """Web検索のモック実装"""
    knowledge_base = {
        "富士山": "富士山（3776m）は日本最高峰。静岡・山梨の県境に位置。世界文化遺産（2013年）。",
        "東京": "東京は日本の首都。人口約1400万人。世界最大の都市圏の一つ。",
        "Python": "Pythonは1991年にGuido van Rossumが開発した汎用プログラミング言語。読みやすさが特徴。",
        "AI": "人工知能（AI）はコンピュータが人間の知的行動を模倣する技術。機械学習・深層学習を含む。",
        "ChatGPT": "ChatGPTはOpenAIが2022年11月に公開したAIチャットボット。GPT-4を使用。",
    }
    for key, value in knowledge_base.items():
        if key.lower() in query.lower():
            return f"検索結果: {value}"
    return f"検索結果: 「{query}」に関する情報が見つかりませんでした。"


# ツール説明（LLMに渡すメタデータ）
TOOLS = [
    {
        "name": "get_weather",
        "description": "指定した都市の現在の天気情報を取得します。日本の主要都市に対応しています。",
        "parameters": {
            "city": {"type": "string", "description": "天気を調べる都市名（例: 東京、大阪、札幌）"}
        },
        "required": ["city"]
    },
    {
        "name": "calculate",
        "description": "数式を計算します。四則演算、三角関数、平方根などに対応。",
        "parameters": {
            "expression": {"type": "string", "description": "計算する数式（例: '2 + 3 * 4', 'sqrt(16)'）"}
        },
        "required": ["expression"]
    },
    {
        "name": "search",
        "description": "キーワードでWeb検索をして情報を取得します。",
        "parameters": {
            "query": {"type": "string", "description": "検索キーワード"}
        },
        "required": ["query"]
    }
]

# ツール名→関数のマッピング
TOOL_FUNCTIONS = {
    "get_weather": get_weather_mock,
    "calculate": calculate,
    "search": search_mock,
}

# 確認
print("利用可能なツール:")
for tool in TOOLS:
    print(f"  - {tool['name']}: {tool['description'][:50]}...")

# 個別ツールのテスト
print("\nツールテスト:")
print(get_weather_mock("東京"))
print(calculate("sqrt(144) + 2**10"))
print(search_mock("Python"))

In [ ]:
# ツールを使うエージェントループ

TOOL_USE_SYSTEM = """
あなたは日本語で応答するアシスタントです。
以下のツールを使用できます:

{tools_description}

ツールを使う場合は、必ず以下のJSON形式で回答してください:
{{"use_tool": true, "tool": "<ツール名>", "args": {{<引数>}}}}

ツールが不要な場合は、通常のテキストで回答してください:
{{"use_tool": false, "answer": "<回答>"}}

重要: 必ず上記のJSON形式のみで回答してください。
"""


def format_tools_description(tools: list) -> str:
    """ツール説明をLLM向けにフォーマット"""
    lines = []
    for tool in tools:
        params = ", ".join(
            f"{k}: {v['description']}"
            for k, v in tool["parameters"].items()
        )
        lines.append(f"- {tool['name']}({params}): {tool['description']}")
    return "\n".join(lines)


def execute_tool(tool_name: str, args: dict) -> str:
    """ツールを実行して結果を返す"""
    if tool_name not in TOOL_FUNCTIONS:
        return f"エラー: ツール '{tool_name}' は存在しません"
    try:
        func = TOOL_FUNCTIONS[tool_name]
        result = func(**args)
        return result
    except Exception as e:
        return f"ツール実行エラー: {str(e)}"


def tool_use_agent(user_query: str, max_steps: int = 5, verbose: bool = True) -> str:
    """
    Tool Useエージェント
    LLMがツールを必要に応じて呼び出しながら質問に答える
    """
    tools_desc = format_tools_description(TOOLS)
    system = TOOL_USE_SYSTEM.format(tools_description=tools_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query}
    ]

    if verbose:
        print(f"ユーザー: {user_query}")
        print("\nエージェントの思考プロセス:")
        print("-" * 50)

    for step in range(max_steps):
        response = llm_call(messages)

        # JSONをパース
        try:
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
            else:
                parsed = {"use_tool": False, "answer": response}
        except Exception:
            parsed = {"use_tool": False, "answer": response}

        if parsed.get("use_tool") and OLLAMA_AVAILABLE:
            # ツールを呼び出す
            tool_name = parsed.get("tool", "")
            tool_args = parsed.get("args", {})

            if verbose:
                print(f"ステップ {step+1}: ツール呼び出し → {tool_name}({tool_args})")

            tool_result = execute_tool(tool_name, tool_args)

            if verbose:
                print(f"  ツール結果: {tool_result}")

            # ツール結果をメッセージに追加
            messages.append({"role": "assistant", "content": response})
            messages.append({
                "role": "user",
                "content": f"ツール実行結果: {tool_result}\n\nこの結果を使って質問に答えてください。"
            })

        else:
            # 最終回答
            final_answer = parsed.get("answer", response)
            if verbose:
                print(f"ステップ {step+1}: 最終回答")
                print("-" * 50)
                print(f"\n最終回答: {final_answer}")
            return final_answer

    # max_steps到達
    return "最大ステップ数に達しました。"


print("エージェントループを定義しました")

In [ ]:
# エージェントの実行例

print("=" * 60)
print("テスト1: 天気情報")
print("=" * 60)
result1 = tool_use_agent("東京の天気は今日どうですか？")

print("\n" + "=" * 60)
print("テスト2: 計算")
print("=" * 60)
result2 = tool_use_agent("15の2乗に100を足した値を計算してください。")

print("\n" + "=" * 60)
print("テスト3: 検索")
print("=" * 60)
result3 = tool_use_agent("Pythonプログラミング言語について教えてください。")

## 2. ReActパターン（Reasoning + Acting）

**ReAct** は「思考（Reasoning）」と「行動（Acting）」を交互に行うパターンです。
各ステップで「なぜこの行動をとるか」を明示的に推論するため、デバッグがしやすくなります。

```
Thought: 東京の天気を調べる必要がある
Action: get_weather({"city": "東京"})
Observation: 東京の天気: 晴れ, 22℃, 湿度55%
Thought: 天気情報が取得できた。回答を生成できる
Answer: 東京は現在晴れで、気温22℃です。
```

In [ ]:
# ReActプロンプトテンプレート

REACT_SYSTEM_PROMPT = """
あなたはステップバイステップで問題を解くアシスタントです。
以下の形式で思考と行動を交互に行ってください:

利用可能なツール:
{tools}

必ず以下の形式で回答してください（JSONブロック内に記述）:

```json
{{"thought": "<何を考えているか>", "action": "<ツール名またはnone>", "action_input": {{<引数>}}, "is_final": false}}
```

最終回答の場合:
```json
{{"thought": "<最終的な考え>", "action": "none", "action_input": {{}}, "is_final": true, "answer": "<最終回答>"}}
```

重要:
- 一度に一つのアクションのみ実行
- ツール結果を受け取ったら次の思考ステップへ
- is_final=trueのとき、answerフィールドに最終回答を記述
"""


def react_agent(user_query: str, max_steps: int = 8, verbose: bool = True) -> str:
    """
    ReActエージェント
    Thought → Action → Observation を繰り返す
    """
    tools_desc = "\n".join(
        f"- {t['name']}: {t['description']}" for t in TOOLS
    )
    system = REACT_SYSTEM_PROMPT.format(tools=tools_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"タスク: {user_query}"}
    ]

    if verbose:
        print(f"タスク: {user_query}")
        print("\n" + "=" * 60)

    for step in range(max_steps):
        response = llm_call(messages, temperature=0.1)

        # JSONを抽出してパース
        try:
            json_match = re.search(r'```json\s*(\{.*?\})\s*```', response, re.DOTALL)
            if not json_match:
                json_match = re.search(r'\{.*\}', response, re.DOTALL)
            parsed = json.loads(json_match.group(1) if '```' in response else json_match.group())
        except Exception:
            parsed = {"thought": response, "action": "none", "action_input": {}, "is_final": True, "answer": response}

        thought = parsed.get("thought", "")
        action = parsed.get("action", "none")
        action_input = parsed.get("action_input", {})
        is_final = parsed.get("is_final", False)

        if verbose:
            print(f"ステップ {step+1}:")
            print(f"  Thought: {thought}")

        if is_final or action == "none":
            final_answer = parsed.get("answer", thought)
            if verbose:
                print(f"  Action: 完了")
                print("=" * 60)
                print(f"\n最終回答: {final_answer}")
            return final_answer

        # ツール実行
        if verbose:
            print(f"  Action: {action}({action_input})")

        observation = execute_tool(action, action_input)

        if verbose:
            print(f"  Observation: {observation}")
            print()

        # 観察結果をメッセージに追加
        messages.append({"role": "assistant", "content": response})
        messages.append({
            "role": "user",
            "content": f"Observation: {observation}\n次のステップを実行してください。"
        })

    return "最大ステップ数に達しました"


print("ReActエージェントを定義しました")

In [ ]:
# ReActエージェントの実行

print("ReActパターン: 単純な質問")
result = react_agent("東京の天気を教えて、そして今日の気温（℃）の2乗を計算してください。")

In [ ]:
# マルチステップの例（計算 + 検索）

print("ReActパターン: マルチステップ")
print("=" * 60)

multi_step_query = """
以下の手順で回答してください:
1. 富士山について検索する
2. 富士山の標高（3776m）と東京スカイツリーの高さ（634m）の差を計算する
3. 結果をまとめて回答する
"""

result = react_agent(multi_step_query)

## 3. メモリと状態管理

長い会話や複数回のタスク実行では、過去の情報を記憶する**メモリ機能**が重要です。

```
短期メモリ（会話バッファ）  長期メモリ（要約/ベクターDB）
┌──────────────────────┐  ┌────────────────────────────┐
│ User: 私の名前は山田  │  │ [要約] ユーザーは山田さん。 │
│ AI: わかりました     │  │ Python初心者。東京在住。    │
│ User: Pythonを学びたい│  └────────────────────────────┘
│ AI: いい目標ですね   │
│ ...（増え続ける）    │
└──────────────────────┘
  制限: コンテキスト長の上限
```

In [ ]:
# メモリ付きエージェント

@dataclass
class ConversationMemory:
    """会話メモリを管理するクラス"""
    max_messages: int = 20         # 最大メッセージ数
    max_tokens_estimate: int = 3000  # 概算トークン上限
    messages: list = field(default_factory=list)
    summary: str = ""              # 古い会話の要約

    def add_message(self, role: str, content: str):
        """メッセージを追加し、上限を超えたら要約する"""
        self.messages.append({"role": role, "content": content})

        # トークン数を簡易推定（文字数 / 3）
        total_chars = sum(len(m["content"]) for m in self.messages)
        if total_chars > self.max_tokens_estimate * 3 or len(self.messages) > self.max_messages:
            self._summarize_and_truncate()

    def _summarize_and_truncate(self):
        """古い会話を要約して圧縮する"""
        if len(self.messages) < 4:
            return

        # 古い会話の半分を要約対象に
        half = len(self.messages) // 2
        old_messages = self.messages[:half]
        self.messages = self.messages[half:]

        # 要約プロンプト
        summary_text = "\n".join(
            f"{m['role']}: {m['content'][:100]}" for m in old_messages
        )
        summary_prompt = f"""
以下の会話を3文以内で要約してください。重要な事実（名前、数値、決定事項）は必ず含めてください:

{summary_text}

要約:"""

        if OLLAMA_AVAILABLE:
            new_summary = llm_call([{"role": "user", "content": summary_prompt}])
        else:
            new_summary = f"[要約] {len(old_messages)}件のメッセージを圧縮しました。"

        # 既存の要約と結合
        if self.summary:
            self.summary = f"{self.summary}\n{new_summary}"
        else:
            self.summary = new_summary

        print(f"  [メモリ圧縮] {half}件のメッセージを要約しました")

    def get_messages_with_context(self, system: str = "") -> list:
        """システムメッセージ + 要約 + 現在のメッセージを結合して返す"""
        result = []
        if system:
            context = system
            if self.summary:
                context += f"\n\n## 過去の会話の要約\n{self.summary}"
            result.append({"role": "system", "content": context})
        elif self.summary:
            result.append({"role": "system", "content": f"過去の会話の要約:\n{self.summary}"})

        result.extend(self.messages)
        return result


def memory_agent(system_prompt: str = ""):
    """メモリ付きエージェントのファクトリ関数"""
    memory = ConversationMemory(max_messages=10, max_tokens_estimate=1000)

    def chat(user_input: str) -> str:
        memory.add_message("user", user_input)
        messages = memory.get_messages_with_context(system_prompt)
        response = llm_call(messages)
        memory.add_message("assistant", response)
        return response

    return chat, memory


# テスト
print("メモリ付きエージェントのテスト")
print("=" * 60)

agent_chat, agent_memory = memory_agent(
    "あなたは親切な日本語のアシスタントです。ユーザーの情報を覚えて会話してください。"
)

conversations = [
    "こんにちは！私の名前は田中太郎です。",
    "私はPythonプログラマーで、AIに興味があります。",
    "好きな食べ物はラーメンです。",
    "私の名前を覚えていますか？"  # メモリテスト
]

for user_msg in conversations:
    print(f"\nユーザー: {user_msg}")
    response = agent_chat(user_msg)
    print(f"AI: {response[:200]}")

print(f"\n現在のメモリ状態:")
print(f"  メッセージ数: {len(agent_memory.messages)}")
print(f"  要約: {agent_memory.summary[:100] if agent_memory.summary else 'なし'}")

## 4. マルチエージェントパターン

複数の専門エージェントが協力してタスクをこなします。

```
             ┌────────────────────┐
             │  オーケストレーター  │
             │ (タスク割り当て)    │
             └──────────┬─────────┘
                        │
            ┌───────────┴───────────┐
            │                       │
   ┌────────────────┐    ┌─────────────────┐
   │  リサーチャー   │    │    ライター      │
   │ (情報収集専門) │    │ (文章作成専門)  │
   └────────────────┘    └─────────────────┘
            │                       │
            └───────────┬───────────┘
                        │
                   最終出力
```

In [ ]:
# マルチエージェントシステムの実装

@dataclass
class AgentMessage:
    """エージェント間のメッセージ"""
    sender: str
    content: str
    message_type: str = "text"  # text, research, draft, final


class ResearcherAgent:
    """情報収集専門エージェント"""
    name = "リサーチャー"
    system_prompt = """
あなたは情報収集の専門家です。与えられたトピックについて、
利用可能なツールを使って事実情報を収集し、箇条書きで報告してください。

報告形式:
## 調査結果: [トピック]
- 事実1
- 事実2
- 事実3
（具体的な数値や情報を含める）
"""

    def research(self, topic: str) -> AgentMessage:
        """トピックについて調査する"""
        print(f"\n[{self.name}] '{topic}' を調査中...")

        # 検索ツールを使って情報収集
        search_result = search_mock(topic)

        messages = [
            {"role": "system", "content": self.system_prompt},
            {
                "role": "user",
                "content": f"'{topic}'について調査してください。\n\n検索結果: {search_result}"
            }
        ]

        if OLLAMA_AVAILABLE:
            result = llm_call(messages)
        else:
            result = f"""## 調査結果: {topic}
- [デモ] {search_result}
- AIは1956年にダートマス会議で命名された
- 機械学習、深層学習、自然言語処理などが主要分野
- ChatGPTは2022年11月に公開され、1億ユーザーを最短で達成"""

        print(f"[{self.name}] 調査完了")
        return AgentMessage(sender=self.name, content=result, message_type="research")


class WriterAgent:
    """文章作成専門エージェント"""
    name = "ライター"
    system_prompt = """
あなたはプロのライターです。提供された調査結果をもとに、
読みやすく魅力的な記事を日本語で作成してください。

記事の条件:
- タイトル、導入部、本文、まとめを含める
- 読者が興味を持てるような書き方
- 事実に基づいた正確な内容
- 400-600文字程度
"""

    def write(self, research: AgentMessage, request: str) -> AgentMessage:
        """調査結果をもとに記事を作成する"""
        print(f"\n[{self.name}] 記事を執筆中...")

        messages = [
            {"role": "system", "content": self.system_prompt},
            {
                "role": "user",
                "content": f"リクエスト: {request}\n\n調査データ:\n{research.content}"
            }
        ]

        if OLLAMA_AVAILABLE:
            article = llm_call(messages, temperature=0.7)
        else:
            article = f"""# AIの世界へようこそ

[デモ記事] 人工知能（AI）は現代社会を大きく変えつつある技術です。

1956年のダートマス会議で「人工知能」という言葉が生まれてから、約70年が経ちました。
今やAIは医療、自動運転、翻訳など様々な分野で活躍しています。

特に2022年のChatGPT登場以降、AIは私たちの日常に急速に浸透しています。

AIの発展は続き、私たちの生活をさらに豊かにしてくれるでしょう。"""

        print(f"[{self.name}] 執筆完了")
        return AgentMessage(sender=self.name, content=article, message_type="final")


print("マルチエージェントクラスを定義しました")
print(f"  エージェント1: {ResearcherAgent.name}")
print(f"  エージェント2: {WriterAgent.name}")

In [ ]:
# オーケストレーターとマルチエージェント実行

def orchestrator(task: str, verbose: bool = True) -> str:
    """
    オーケストレーター: タスクを分解し、適切なエージェントに割り当てる
    """
    if verbose:
        print(f"オーケストレーター: タスク受信")
        print(f"タスク: {task}")
        print("=" * 60)

    # ステップ1: トピックを特定
    topic_prompt = f"""以下のタスクから調査すべきトピックを1〜3語で抽出してください。
回答はトピックのみ（例: "AI", "東京の歴史", "Python"）:

タスク: {task}"""

    if OLLAMA_AVAILABLE:
        topic = llm_call([{"role": "user", "content": topic_prompt}]).strip()
    else:
        # デモ: タスクからキーワードを抽出
        topic = "AI"

    if verbose:
        print(f"\n[オーケストレーター] 調査トピック: {topic}")

    # ステップ2: リサーチャーに調査依頼
    researcher = ResearcherAgent()
    research_result = researcher.research(topic)

    if verbose:
        print(f"\n調査結果プレビュー:")
        print(research_result.content[:300] + "...")

    # ステップ3: ライターに執筆依頼
    writer = WriterAgent()
    article = writer.write(research_result, task)

    if verbose:
        print("\n" + "=" * 60)
        print("最終成果物:")
        print("=" * 60)
        print(article.content)

    return article.content


# 実行
task = "AIについての短い記事を書いてください。最新のトレンドも含めてください。"
final_article = orchestrator(task)

## 5. エージェントの評価と制限

In [ ]:
# エージェントの安全チェック・トークンカウント・コスト推定

# 1. 安全チェック関数
UNSAFE_PATTERNS = [
    r'os\.system',
    r'subprocess',
    r'exec\s*\(',
    r'eval\s*\(',
    r'__import__',
    r'open\s*\([^)]*[wWaA]',  # ファイル書き込み
    r'rm\s+-rf',
    r'DROP\s+TABLE',          # SQLインジェクション
    r'<script',                # XSS
]


def safety_check(text: str) -> dict:
    """
    LLM出力の安全チェック
    潜在的に危険なパターンを検出する
    """
    issues = []
    for pattern in UNSAFE_PATTERNS:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            issues.append(f"危険パターン検出: {pattern} ({matches[0]}...)")

    # 異常に長い出力チェック
    if len(text) > 50000:
        issues.append(f"出力が異常に長い: {len(text)}文字")

    return {
        "is_safe": len(issues) == 0,
        "issues": issues,
        "text_length": len(text)
    }


# 2. トークンカウント（簡易推定）
def estimate_tokens(text: str, language: str = "japanese") -> int:
    """
    テキストのトークン数を簡易推定
    日本語: 1文字 ≈ 1-2トークン
    英語: 1単語 ≈ 1.3トークン
    """
    if language == "japanese":
        # 日本語は文字数がほぼトークン数
        japanese_chars = len(re.findall(r'[\u3000-\u9fff]', text))
        other_chars = len(text) - japanese_chars
        return japanese_chars + (other_chars // 4)
    else:
        words = len(text.split())
        return int(words * 1.3)


# 3. コスト推定
# 2024年時点の参考価格（実際は変動します）
PRICING = {
    "gpt-4o": {"input": 5.0, "output": 15.0},        # $5/$15 per 1M tokens
    "gpt-4o-mini": {"input": 0.15, "output": 0.6},   # $0.15/$0.6 per 1M tokens
    "claude-3-5-sonnet": {"input": 3.0, "output": 15.0},
    "ollama-local": {"input": 0.0, "output": 0.0},    # 無料！
}


def estimate_cost(input_tokens: int, output_tokens: int, model: str = "gpt-4o") -> dict:
    """
    API呼び出しコストを推定
    """
    if model not in PRICING:
        return {"error": f"モデル '{model}' の価格情報がありません"}

    price = PRICING[model]
    input_cost = (input_tokens / 1_000_000) * price["input"]
    output_cost = (output_tokens / 1_000_000) * price["output"]
    total_cost = input_cost + output_cost

    return {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "input_cost_usd": round(input_cost, 6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd": round(total_cost, 6),
        "total_cost_jpy": round(total_cost * 150, 2),  # 1USD = 150JPY
    }


# テスト
test_outputs = [
    "これは安全なテキストです。Pythonの基本的な使い方を説明します。",
    "import os; os.system('rm -rf /')",  # 危険なコード（テスト用）
]

print("=== 安全チェックテスト ===")
for text in test_outputs:
    result = safety_check(text)
    status = "安全" if result["is_safe"] else "危険"
    print(f"テキスト: {text[:50]}")
    print(f"  → {status}: {result['issues'] if result['issues'] else '問題なし'}")

print("\n=== トークン・コスト推定 ===")
sample_text = "人工知能（AI）は現代社会を大きく変えつつある技術です。機械学習や深層学習を基盤として、様々な分野で活用されています。"
token_count = estimate_tokens(sample_text)
print(f"サンプルテキスト: {len(sample_text)}文字 ≈ {token_count}トークン")

print("\nモデル別コスト比較（入力1000トークン、出力500トークン）:")
for model_name in PRICING:
    cost = estimate_cost(1000, 500, model_name)
    print(f"  {model_name:<25}: ${cost['total_cost_usd']:.6f} (¥{cost['total_cost_jpy']:.3f})")

## まとめ：エージェントの現状と将来

### 実装したパターンのまとめ

| パターン | 特徴 | 適したユースケース |
|---------|------|-------------------|
| Tool Use | LLMが外部ツールを呼び出す | 検索、計算、API連携 |
| ReAct | Thought→Action→Observationループ | 複雑な推論タスク |
| Memory Agent | 会話履歴の管理・圧縮 | 長期会話、パーソナライズ |
| Multi-Agent | 専門エージェントの分業 | 複雑なワークフロー |

### エージェントの現状の課題

```
課題1: 信頼性
  エラーが連鎖しやすい（1ステップの失敗が全体に波及）
  → リトライ機構、エラーハンドリングが重要

課題2: コスト
  多ステップ = 多トークン消費
  → 適切なモデル選択、キャッシュ戦略が必要

課題3: 評価の難しさ
  「エージェントが正しく動いたか」を自動評価するのが難しい
  → 専用のエージェント評価フレームワークが開発中

課題4: 安全性
  プロンプトインジェクション攻撃のリスク
  → 入力・出力の検証、サンドボックス実行環境が必要
```

### 将来の展望

1. **Long-context models**: コンテキスト長の増大でメモリ管理が簡略化
2. **Nativeツール呼び出し**: OpenAI/Anthropicの関数呼び出しAPIでより安定した Tool Use
3. **Computer Use**: PCの画面を直接操作するエージェント（Claude Computer Use等）
4. **エージェントフレームワーク**: LangGraph, AutoGen, CrewAIなどのフレームワーク成熟

### 次のステップ

- **LangChain/LangGraph**: プロダクション品質のエージェント構築ライブラリ
- **OpenAI Assistants API**: ツール・ファイル・メモリ管理をマネージドサービスで
- **Computer Use (Claude)**: ブラウザ・デスクトップを操作するエージェント
- **05_llm_evaluation**: エージェントの品質を自動評価する方法

### 参考リンク

- [ReAct論文 (Yao et al., 2022)](https://arxiv.org/abs/2210.03629)
- [LangGraph ドキュメント](https://langchain-ai.github.io/langgraph/)
- [AutoGen (Microsoft)](https://github.com/microsoft/autogen)
- [Anthropic Computer Use](https://docs.anthropic.com/claude/docs/computer-use)